# 📊 DATA COLLECTION & CLEANING — AI Tech vs Traditional (S&P 500)

**Dataset:** S&P 500 Tech (2010-nay) + Tech Top 50 + Financials + Market Cap Scatter  
**Ngôn ngữ:** Python | **Môi trường:** Jupyter Notebook

---

## 1. 🎯 MUC TIEU GIAI DOAN 1

- Tai va lam sach du lieu goc tu 4 file CSV
- Chuan hoa dinh dang (header, kieu du lieu, missing values)
- Chuyen doi S&P 500 daily data tu wide -> long
- Ghep them chi so tai chinh va chi so market cap de san sang phan tich

---

## 2. 📂 DU LIEU DAU VAO

| File | Mo ta |
|------|-------|
| `SnP_daily_update.csv` | Gia Close/Open/High/Low/Volume hang ngay cho 103 ticker |
| `Top 50 US Tech Companies 2022 - 2023.csv` | Thong tin 50 cong ty tech (Revenue, Market Cap, Employee Size) |
| `constituents-financials.csv` | Chi so tai chinh (P/E, Dividend Yield, EBITDA, Market Cap, ...) |
| `scatter-data.csv` | Market cap va P/E ratio cho nhieu cong ty |

**Cac cot chinh mong doi**: `date`, `ticker`, `sector`, `daily_return`, `market_cap`, `volume`, `pe_ratio`.

---

## 3. 🗺️ QUY TRINH XU LY (STAGE 1)

```
1) Load & Clean S&P 500 Daily Data
   - Doc file, chuan hoa header 2 dong
   - Reshape wide -> long
   - Tinh daily_return
2) Load & Clean Tech Companies
   - Chuan hoa column, xu ly missing values
   - Gan nhan AI-focused vs Traditional
3) Merge Financial Data
   - Merge constituents-financials + scatter-data
```

---

## 4. 🛠️ THU VIEN SU DUNG

| Nhom | Thu vien | Muc dich |
|------|----------|----------|
| Xu ly du lieu | `pandas`, `numpy` | Doc CSV, lam sach, reshape, merge |

---

## 5. 📦 OUTPUT GIAI DOAN 1

- `snp_long`: S&P 500 daily data da reshape va tinh `daily_return`
- `tech_df`: Tech companies da duoc gan nhan `ai_group`
- `fin_merged`: Financials da duoc ghep voi scatter-data

---

## 6. ⚙️ SETUP & DATA COLLECTION

In [1]:
# Import thu vien
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print('✅ Import thanh cong!')

✅ Import thanh cong!


In [2]:
# 1) Load & clean S&P 500 daily data, reshape wide -> long
snp_raw = pd.read_csv('dataset/SnP_daily_update.csv', header=[0, 1])

# Normalize 2-row header and fix Date column
snp_raw.columns = [(str(a), str(b)) for a, b in snp_raw.columns]
snp_raw = snp_raw.rename(columns={('Price', 'Ticker'): 'Date'})
snp_raw = snp_raw[snp_raw['Date'].astype(str).str.lower() != 'date']
snp_raw['Date'] = pd.to_datetime(snp_raw['Date'], errors='coerce')
snp_raw = snp_raw.dropna(subset=['Date'])

# Build long format for price metrics
metrics = ['Close', 'Open', 'High', 'Low', 'Volume']
long_frames = []
for metric in metrics:
    cols = [c for c in snp_raw.columns if isinstance(c, tuple) and c[0] == metric]
    if not cols:
        continue
    metric_df = snp_raw[['Date'] + cols].copy()
    metric_df.columns = ['Date'] + [c[1] for c in cols]
    metric_long = metric_df.melt('Date', var_name='ticker', value_name=metric.lower())
    long_frames.append(metric_long)

snp_long = long_frames[0]
for metric_long in long_frames[1:]:
    snp_long = snp_long.merge(metric_long, on=['Date', 'ticker'], how='left')

# Compute daily returns from close prices
snp_long = snp_long.sort_values(['ticker', 'Date'])
snp_long['daily_return'] = snp_long.groupby('ticker')['close'].pct_change()

print('✅ S&P 500 long data ready:', snp_long.shape)
snp_long.head()

✅ S&P 500 long data ready: (424051, 8)


,Date,ticker,close,open,high,low,volume,daily_return
0,2010-01-04,AAPL,6.406479,6.389117,6.421148,6.357685,493729600.0,NaN
1,2010-01-05,AAPL,6.417556,6.424142,6.453778,6.383729,601904800.0,0.001729
2,2010-01-06,AAPL,6.315477,6.417557,6.443002,6.308891,552160000.0,-0.015906
3,2010-01-07,AAPL,6.303802,6.338827,6.346311,6.258001,477131200.0,-0.001849
4,2010-01-08,AAPL,6.345710,6.295418,6.346309,6.258299,447610800.0,0.006648


In [3]:
# 2) Load & clean Tech Companies, classify AI-focused vs Traditional
tech_df = pd.read_csv('dataset/Top 50 US Tech Companies 2022 - 2023.csv')
tech_df = tech_df.dropna(subset=['Company Name', 'Stock Name'])
tech_df['Employee Size'] = pd.to_numeric(tech_df['Employee Size'], errors='coerce')

ai_focused = {
    'NVDA', 'MSFT', 'GOOG', 'META', 'AMD', 'AMZN', 'CRM', 'PLTR', 'SNPS', 'CDNS',
    'NOW', 'CRWD', 'PANW', 'ADBE', 'INTC', 'ORCL', 'ANET'
}
traditional = {
    'AAPL', 'CSCO', 'HPE', 'HPQ', 'IBM', 'DELL', 'BBY', 'GLW', 'STX', 'WDC', 'SWKS',
    'TER', 'KEYS', 'MCHP', 'QCOM', 'TXN', 'KLAC', 'LRCX', 'AMAT', 'FFIV', 'AKAM',
    'GEN', 'VRSN', 'FISV', 'ADP', 'BR', 'NDAQ', 'ICE', 'CME', 'CBOE'
}

def ai_group_from_ticker(ticker):
    if ticker in ai_focused:
        return 'AI-focused'
    if ticker in traditional:
        return 'Traditional'
    return 'Unclassified'

tech_df['ai_group'] = tech_df['Stock Name'].apply(ai_group_from_ticker)
print('✅ Tech companies ready:', tech_df.shape)
tech_df.head()

✅ Tech companies ready: (50, 11)


,Company Name,Industry,Sector,HQ State,Founding Year,Annual Revenue 2022-2023 (USD in Billions),Market Cap (USD in Trillions),Stock Name,Annual Income Tax in 2022-2023 (USD in Billions),Employee Size,ai_group
0,Apple Inc.,Technology,Consumer Electronics,California,1976,387.53,2.520,AAPL,18.314,164000,Traditional
1,Microsoft Corporation,Technology,Software Infrastructure,Washington,1975,204.09,2.037,MSFT,15.139,221000,AI-focused
2,Alphabet (Google),Technology,Software Infrastructure,California,1998,282.83,1.350,GOOG,11.356,190234,AI-focused
3,Amazon,Technology,Software Application,Washington,1994,513.98,1.030,AMZN,-3.217,1541000,AI-focused
4,NVIDIA Corporation,Technology,Semiconductors,California,1993,26.97,0.653,NVDA,0.189,22473,AI-focused


In [4]:
# 3) Merge financial data: constituents-financials + scatter-data
fin_df = pd.read_csv('dataset/constituents-financials.csv')
scatter_df = pd.read_csv('dataset/scatter-data.csv')

# Clean numeric columns in financials
num_cols = fin_df.select_dtypes(include=np.number).columns
fin_df[num_cols] = fin_df[num_cols].fillna(fin_df[num_cols].median())

# Normalize company name to merge with scatter-data (company, pe_ratio, market_cap_b)
def normalize_name(value):
    if not isinstance(value, str):
        return ''
    return ''.join(ch.lower() for ch in value if ch.isalnum())

fin_df['company_key'] = fin_df['Name'].apply(normalize_name)
scatter_df['company_key'] = scatter_df['company'].apply(normalize_name)
fin_merged = fin_df.merge(scatter_df, on='company_key', how='left', suffixes=('', '_scatter'))

print('✅ Financial merged data ready:', fin_merged.shape)
fin_merged.head()

✅ Financial merged data ready: (503, 19)


,Symbol,Name,Sector,Price,Price/Earnings,Dividend Yield,Earnings/Share,52 Week Low,52 Week High,Market Cap,EBITDA,Price/Sales,Price/Book,SEC Filings,company_key,company,sector,market_cap_b,pe_ratio
0,MMM,3M,Industrial Conglomerates,146.22,28.173410,0.0213,5.19,139.34,177.41,7.626356e+10,6.240000e+09,3.047617,23.372763,http://www.sec.gov/cgi-bin/browse-edgar?action...,3m,3M,Industrial Conglomerates,76.26,28.17
1,AOS,A. O. Smith,Building Products,56.01,14.936000,0.0245,3.75,55.98,81.87,7.719781e+09,7.953000e+08,2.025179,4.110825,http://www.sec.gov/cgi-bin/browse-edgar?action...,aosmith,A. O. Smith,Building Products,7.72,14.94
2,ABT,Abbott Laboratories,Health Care Equipment,84.47,23.661066,0.0298,3.57,81.97,139.06,1.471310e+11,1.174400e+10,3.259870,2.826124,http://www.sec.gov/cgi-bin/browse-edgar?action...,abbottlaboratories,Abbott Laboratories,Health Care Equipment,147.13,23.66
3,ABBV,AbbVie,Biotechnology,210.39,103.132355,0.0328,2.04,180.25,244.81,3.717155e+11,2.991500e+10,5.917247,-55.850810,http://www.sec.gov/cgi-bin/browse-edgar?action...,abbvie,NaN,NaN,NaN,NaN
4,ACN,Accenture,IT Consulting & Other Services,168.82,13.826372,0.0386,12.21,155.82,322.86,1.038978e+11,1.273531e+10,1.440817,3.325716,http://www.sec.gov/cgi-bin/browse-edgar?action...,accenture,Accenture,IT Consulting & Other Services,103.90,13.83


In [5]:
# 4) Luu du lieu da lam sach de tai su dung
import os

output_dir = 'dataset/processed'
os.makedirs(output_dir, exist_ok=True)

snp_long.to_csv(os.path.join(output_dir, 'snp_long.csv'), index=False)
tech_df.to_csv(os.path.join(output_dir, 'tech_df.csv'), index=False)
fin_merged.to_csv(os.path.join(output_dir, 'fin_merged.csv'), index=False)

print('✅ Saved:', output_dir)

✅ Saved: dataset/processed
